# 01 — kvpress Fork Setup

This notebook prepares the environment to use a development version of
kvpress from a fork, overriding the released version installed by
`00_setup_check.ipynb`.

Run this once before executing `02_kvpress_niah.ipynb` (or any notebook
that uses kvpress) when you need unreleased changes from your fork.

Unlike the vLLM fork setup (`03`), no compiled extensions need copying —
kvpress is pure Python, so `sys.path.insert` is all that's needed.

## Configuration

In [ ]:
FORK_URL = "https://github.com/fax4ever/kvpress.git"
FORK_BRANCH = "padded-tensor"
FORK_DIR = "/opt/app-root/src/kvpress-fork"

## Step 1 — Clone the fork

In [2]:
import os
import shutil

if os.path.exists(FORK_DIR):
    shutil.rmtree(FORK_DIR)
    print(f"Removed previous clone at {FORK_DIR}")

!git clone {FORK_URL} {FORK_DIR}
!cd {FORK_DIR} && git checkout {FORK_BRANCH}
print(f"\nCloned fork to {FORK_DIR} on branch {FORK_BRANCH}")

Cloning into '/opt/app-root/src/kvpress-fork'...
remote: Enumerating objects: 1544, done.
remote: Counting objects: 100% (767/767), done.
remote: Compressing objects: 100% (239/239), done.
remote: Total 1544 (delta 642), reused 528 (delta 528), pack-reused 777 (from 1)
Receiving objects: 100% (1544/1544), 6.87 MiB | 22.48 MiB/s, done.
Resolving deltas: 100% (1051/1051), done.
branch 'filtering' set up to track 'origin/filtering'.
Switched to a new branch 'filtering'

Cloned fork to /opt/app-root/src/kvpress-fork on branch filtering


## Step 2 — Verify the fork is importable

In [3]:
import sys
sys.path.insert(0, FORK_DIR)

import kvpress
print(f"kvpress location: {os.path.dirname(kvpress.__file__)}")

assert FORK_DIR in kvpress.__file__, (
    f"ERROR: kvpress is NOT loading from the fork!\n"
    f"  Expected path under: {FORK_DIR}\n"
    f"  Actual path:         {kvpress.__file__}"
)
print("\n✓  Fork is active.")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


kvpress location: /opt/app-root/src/kvpress-fork/kvpress

✓  Fork is active.


## Step 3 — Smoke test

In [4]:
from kvpress import KeyDiffPress, BlockPress, PrefillDecodingPress, CompressionRatioDecodingPress

press = PrefillDecodingPress(
    prefilling_press=BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128),
    decoding_press=CompressionRatioDecodingPress(
        base_press=KeyDiffPress(), target_compression_ratio=0.5,
    ),
)

print(f"PrefillDecodingPress: {press}")
print(f"  prefilling: {press.prefilling_press}")
print(f"  decoding:   {press.decoding_press}")
print("\n✓  Press types from fork instantiated successfully.")

PrefillDecodingPress: PrefillDecodingPress(prefilling_press=BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128), decoding_press=CompressionRatioDecodingPress(base_press=KeyDiffPress(compression_ratio=0.0), compression_interval=512, target_size=1, hidden_states_buffer_size=256, target_compression_ratio=0.5))
  prefilling: BlockPress(press=KeyDiffPress(compression_ratio=0.5), block_size=128)
  decoding:   CompressionRatioDecodingPress(base_press=KeyDiffPress(compression_ratio=0.0), compression_interval=512, target_size=1, hidden_states_buffer_size=256, target_compression_ratio=0.5)

✓  Press types from fork instantiated successfully.


## Done

kvpress is now loading from the fork. The same `sys.path.insert` line
must be added at the top of `02_kvpress_niah.ipynb` (before any
`import kvpress`) for the fork to be used during the benchmark.

If the workbench pod is recreated, re-run this notebook after
`00_setup_check.ipynb` to restore the fork clone.